In [ ]:
from agents import Agent, Runner, function_tool ,ItemHelpers
# Runner는 while true loop를 처리하는것으로 main agent와 input을 받는다. 
# Runner가 openAI에 request를 보내면 openai가 응답하고, runner는 그 response를 추출해서 parsing하고, 호출해야 할 tool이 있는지 확인함 
# Runner가 그 tool들을 호출하고, 두 결과르 ㄹ다시 openai에 보냄 
# 그리고 Runner는 final response를 받아서 우리에게 보내준다. 

@function_tool
def get_weather(city:str):
    """Get weather by city""" 
    return "30 degrees"

agent = Agent(
    name = "Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions",
    tools =[get_weather]
)

# run : 입력이 완료되면 보여줌 / run_streamed : 실시간으로 보여줌(지금 지피티처럼)
stream = Runner.run_streamed(agent, "Hello how are you? what is the weather in the capital of Spain?")

async for event in stream.stream_events():

    if event.type == "raw_response_event":
        continue
    elif event.type == "agent_updated_stream_event":
        print("Agent updated to", event.new_agent.name)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(event.item.raw_item.to_dict())
        elif event.item.type == "tool_call_output_item":
            print(event.item.output)
        elif event.item.type == "message_output_item":
            print(ItemHelpers.text_message_output(event.item)) #  response에서 text 추출 해줌 
    print("=" * 20)

Agent updated to Assistant Agent
{'arguments': '{"city":"Madrid"}', 'call_id': 'call_S0IBxSeCJuevGMHyu4KRLF9x', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_069873d5464d790700694cb5481b7c819291b4f5fc9fe02b34', 'status': 'completed'}
Madrid
30 degrees
Hello! I'm doing well, thank you for asking. The weather in Madrid, the capital of Spain, is currently 30 degrees. Let me know if you need more details or anything else!


# 7.2 Stream Events

In [ ]:
from agents import Agent, Runner, function_tool ,ItemHelpers
# Runner는 while true loop를 처리하는것으로 main agent와 input을 받는다. 
# Runner가 openAI에 request를 보내면 openai가 응답하고, runner는 그 response를 추출해서 parsing하고, 호출해야 할 tool이 있는지 확인함 
# Runner가 그 tool들을 호출하고, 두 결과르 ㄹ다시 openai에 보냄 
# 그리고 Runner는 final response를 받아서 우리에게 보내준다. 

@function_tool
def get_weather(city:str):
    """Get weather by city""" 
    return "30 degrees"

agent = Agent(
    name = "Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions",
    tools =[get_weather]
)

# run : 입력이 완료되면 보여줌 / run_streamed : 실시간으로 보여줌(지금 지피티처럼)
# run_sync : await 대신 사용할 수 있는것이며 동작은 run과 같음 
stream = Runner.run(agent, "Hello how are you? what is the weather in the capital of Spain?")

message = ""
args = ""
async for event in stream.stream_events():

    if event.type == "raw_response_event":
        # print(event.data.type) # 우리가 받는 이벤트 확인 가능 
        event_type = event.data.type 
        if event_type == "response.output_text.delta": #   1. response output text 보여주기
            # print(event.data.delta)
            message += event.data.delta
            print(message)
        elif event_type == "response.function_call_arguments.delta": # 모델이 인수들을 작성하는 과정을 보여줌
           #  print(event.data.delta)
           args += event.data.delta
           print(args)
        elif event_type == "response.completed":
            message = ""
            args = ""
    print("=" * 20)

{"
{"city
{"city":"
{"city":"Madrid
{"city":"Madrid"}
Hello
Hello!
Hello! I'm
Hello! I'm doing
Hello! I'm doing well
Hello! I'm doing well,
Hello! I'm doing well, thank
Hello! I'm doing well, thank you
Hello! I'm doing well, thank you for
Hello! I'm doing well, thank you for asking
Hello! I'm doing well, thank you for asking.
Hello! I'm doing well, thank you for asking. The
Hello! I'm doing well, thank you for asking. The weather
Hello! I'm doing well, thank you for asking. The weather in
Hello! I'm doing well, thank you for asking. The weather in Madrid
Hello! I'm doing well, thank you for asking. The weather in Madrid,
Hello! I'm doing well, thank you for asking. The weather in Madrid, the
Hello! I'm doing well, thank you for asking. The weather in Madrid, the capital
Hello! I'm doing well, thank you for asking. The weather in Madrid, the capital of
Hello! I'm doing well, thank you for asking. The weather in Madrid, the capital of Spain
Hello! I'm doing well, thank you for asking. Th

# 7.3 Session Memory 

In [12]:
from agents import Agent, Runner, function_tool ,SQLiteSession 
# SQLiteSession 에 대화를 저장하여 초반에 만든 메모리 역할을 하게 해줄 수 있고 등등의 다양한 역할 가능 

session = SQLiteSession("user_2", "ai-memory.db") # 식별자 , db_id 

@function_tool
def get_weather(city:str):
    """Get weather by city""" 
    return "30 degrees"

agent = Agent(
    name = "Assistant Agent",
    instructions="You are a helpful assistant. Use tools when needed to answer questions",
    tools =[get_weather]
)


In [19]:

# run : 입력이 완료되면 보여줌 / run_streamed : 실시간으로 보여줌(지금 지피티처럼)
# run_sync : await 대신 사용할 수 있는것이며 동작은 run과 같음 
result = await Runner.run(
    agent, 
    "what was my name again?",
    session = session,
) 

print(result.final_output)

Your name is Sam.


In [ ]:
await session.clear_session()  # 세션 초기화

In [16]:
await session.add_items([
    {"role":"user","content":"My name is sam"}
]) # 메시지 추가 

In [18]:
await session.pop_item()

{'id': 'msg_0e99a221139e2df200694cc8a458b4819f9d502820ef945446',
 'content': [{'annotations': [],
   'text': 'Your name is Sam.',
   'type': 'output_text',
   'logprobs': []}],
 'role': 'assistant',
 'status': 'completed',
 'type': 'message'}

# 7.4 Handoffs 

In [29]:
from agents import Agent, Runner ,SQLiteSession, handoff 
# SQLiteSession 에 대화를 저장하여 초반에 만든 메모리 역할을 하게 해줄 수 있고 등등의 다양한 역할 가능 

session = SQLiteSession("user_2", "ai-memory.db") # 식별자 , db_id 
 
geography_agent = Agent(
    name ="Geography Expert Agent",
    instructions="You are a expert in geography, you are answer questions related to them ",
    # handoff_description :
    # agent 에 대한 설명이다. 하지만 기본적으로 main_agent가 해당 agent가 뭘 하는지 알 수 있도록 쓰이는 것임 
    handoff_description="Use this to answer geography related questions.",
    
)

economics_agent = Agent(
    name ="Economics Expert Agent",
    instructions="You are a expert in economics, you are answer questions related to them ",
    handoff_description="Use this to answer economics related questions.", 
) 

main_agent = Agent(
    name = "Main Agent",
    instructions="You are a user facing agent. Transfer to the agent most capable of answering the user's question.",
    handoffs= [ #위에 두개의 에이전트가 있다고 알려줘야됨
        economics_agent,
        geography_agent
    ]
)


In [30]:
result = await Runner.run(
    main_agent,  # 시작 agent
    "What is the second biggest city in Mongolia",
    session = session,     
) 

print(result.last_agent.name)  # 에이전트 개입햇나 확인 
print(result.final_output)

Main Agent
The second biggest city in Mongolia is **Erdenet**. It is an important industrial city, primarily known for its large copper mine. Ulaanbaatar is the largest city and the capital.
